# Deploying NVIDIA Nemotron 3.5 Lightning with TensorRT-LLM

This notebook will walk you through how to run the NVIDIA Nemotron 3.5 Lightning NVFP4 checkpoint via TensorRT-LLM on a single H100.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference performance on NVIDIA GPUs.

Nemotron 3.5 Lightning is published as two checkpoints:

- **BF16**: [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16)
- **NVFP4**: [`nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4`](https://huggingface.co/nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4)

**This notebook runs our recommended configuration: the NVFP4 checkpoint on a single H100 without speculative decoding.** Configurations for the other tested combinations are at the end, under **Additional configurations**.

**Model size:** 30B total parameters, 3B active (MoE)

Prerequisites for this notebook:
- 1x NVIDIA H100 80GB with recent drivers
- Python 3.10+
- Docker

## Overview

- **Serve** the Nemotron 3.5 Lightning NVFP4 checkpoint on a single H100 using TensorRT-LLM
- **Query the model** through an OpenAI-compatible API
- **Invoke tools** using structured function-calling outputs
- **Tune reasoning depth** by configuring the model's thinking budget
- **Reference commands** for MTP, the BF16 checkpoint, and DGX Spark

## Table of Contents

1. **Decoding options for this model** - Base and MTP
2. **Environment setup** - Container image, client dependencies, and GPU check
   - Launch on NVIDIA Brev
   - Pull the TensorRT-LLM Docker image
   - Install notebook client dependencies
   - Verify GPU
3. **OpenAI-compatible server** - Launch TensorRT-LLM and confirm it is ready
   - Launch the Docker container
   - Create a YAML file
   - Configuration reference
   - Load the model
   - Wait for the server to be ready
4. **Generate responses** - Chat completions, reasoning, and tool calling
   - Client setup
   - Single completion
   - Sequential completions
   - Streamed generation
   - Reasoning
   - Tool calling
   - Controlling reasoning budget
5. **Cleanup and shutdown** - Free the GPU and reset the kernel
6. **Additional configurations** - Reference configurations for other hardware and precisions
   - 1x H100
   - 1x DGX Spark


## Decoding options for this model

Nemotron 3.5 Lightning can produce tokens four ways. All four serve the same weights and differ only in how many tokens come out of a single forward pass. The base option decodes one token per pass - the other three add speculative decoding, where a cheap draft proposes several tokens ahead, the model verifies them all in a single pass, and every token up to the first mismatch is kept. A rejected token invalidates itself and everything after it, so the payoff depends on how often drafts are right - a bad guess costs compute without producing output.

| Option | Where drafts come from | Extra weights to download |
|---|---|---|
| **Base** (no speculative decoding) | nothing, one token per pass | none |
| **MTP** | a prediction layer inside the checkpoint | none |
| **DFlash** | a separate block-diffusion draft model, a whole block per pass | DFlash checkpoint |
| **DSpark** | a separate semi-autoregressive draft model, a whole block per pass | DSpark checkpoint |

TensorRT-LLM supports MTP for this checkpoint. DFlash and DSpark, which draft from a separate model, are covered in the vLLM and SGLang cookbooks instead. Pick exactly one: a server has a single draft path, so the configuration conflicts at launch rather than stacking.

**Concurrency decides whether speculation pays off.** With few requests in flight the GPU has spare capacity, and speculation spends it to shorten the critical path, which lowers per-request latency. Under heavy load the GPU is already busy producing real tokens, so verifying drafts that end up rejected takes throughput away from queued work. `--max_batch_size` caps how many requests can be in flight, so it is the flag that decides which side of that tradeoff you land on.

**Draft length is the main knob.** `max_draft_len` sets how far ahead to guess. Raising it improves the best case per step but lowers the odds that the whole run is accepted, and wastes more compute when it is not - lowering it shrinks the win but makes it more consistent.

**Two caches share the GPU.** Attention layers use a KV cache, set to FP8 here by `kv_cache_config.dtype`, while the Mamba layers keep a fixed-size recurrent state per sequence in FP16 with stochastic rounding. `free_gpu_memory_fraction` bounds how much of the GPU the two may claim in total.

This notebook uses the **Base** configuration.

## Environment setup

### Launch on NVIDIA Brev

You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch the NVFP4 variant on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide.

**For NVFP4 (1x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3HawHhqQ43fhtSFmVFnqAtWCCGN)

### Pull the TensorRT-LLM Docker image

The model runs inside a TensorRT-LLM container. Pull it once before starting the server:

```shell
docker pull nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc24
```

### Install notebook client dependencies

These are for the notebook only: `openai` sends the requests, `transformers` provides the tokenizer used in the reasoning budget section, and `jinja2` renders the chat template it applies.

In [1]:
# Bootstrap pip only if the kernel environment is missing it
import importlib.util, subprocess, sys

if importlib.util.find_spec("pip") is None:
    subprocess.run([sys.executable, "-m", "ensurepip", "--upgrade"], check=True)

%pip install -q openai==2.38.0 transformers==5.9.0 "jinja2>=3.1.0"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Verify GPU

Confirm the GPU is detected correctly on the host.

> **Expected output:** One row per GPU, showing an H100 with roughly 80 GB of memory alongside the host driver version. If `nvidia-smi` is not found, the NVIDIA driver is not installed.

In [2]:
# Confirm the GPU is visible on the host
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

index, name, memory.total [MiB], driver_version
0, NVIDIA H100 PCIe, 81559 MiB, 570.148.08


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using TensorRT-LLM.

### Launch the Docker container

Open a terminal on the host and start an interactive shell inside the TensorRT-LLM container. The `--network=host` flag makes the server reachable at `localhost:8000` from the notebook.

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all \
  --network=host \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc24
```

> **Note:** Mount the HuggingFace cache directory so model weights are read from disk rather than re-downloaded on each run. Replace `~/.cache/huggingface` if your cache is in a different location.

Run the `trtllm-serve` command below from inside this container. The configurations in **Additional configurations** at the end run here too.

### Create a YAML file

Run the following inside that container, before serving. It configures the Marlin MoE backend with an FP8 KV cache and stochastic-rounded Mamba state, and the `trtllm-serve` command in the next section reads it at startup.

```shell
cat > ./extra-llm-api-config.yml << EOF
kv_cache_config:
  dtype: fp8
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.8
  mamba_ssm_cache_dtype: float16
  mamba_ssm_stochastic_rounding: true
  mamba_ssm_philox_rounds: 5
  mamba_state_config:
    periodic_snapshot_interval: 8192

moe_config:
  backend: MARLIN

nvfp4_gemm_config:
  allowed_backends: [marlin, cutlass, cublaslt, cuda_core]

cuda_graph_config:
  enable_padding: true
  max_batch_size: 8

enable_chunked_prefill: true
num_postprocess_workers: 4
print_iter_log: true
stream_interval: 10
disable_overlap_scheduler: false
EOF
```

### Configuration reference

Settings for the configuration this notebook runs.

| Setting | NVFP4 | Why |
|---|---|---|
| **Model** | `nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4` | 4-bit weights fit a 30B MoE on one GPU |
| **Served model name** | `nemotron-3.5-lightning` | What the client cells send |
| **Hardware (this notebook)** | 1x H100 80GB | What these values were tuned on |
| **Docker image** | `nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc24` | Release candidate validated for this model |
| **Remote code** | Trusted | The checkpoint ships its own modeling code |
| **MoE backend** | `MARLIN` | NVFP4 expert path supported on H100 |
| **KV cache dtype** | FP8 | Halves memory per cached token |
| **KV block reuse** | Disabled | Off in the validated configuration |
| **GPU memory fraction** | 0.8 | Caps total cache memory |
| **Mamba state cache** | FP16, stochastic rounding, 5 Philox rounds | Halves state memory |
| **Mamba snapshot interval** | 8192 tokens | Periodic state snapshots for long sequences |
| **Max batch size** | 8 | Caps concurrency and captured graphs |
| **CUDA graph padding** | Enabled | Pads batches up to a captured graph size |
| **Max num tokens** | 8192 | Bounds prefill work per step |
| **Max sequence length** | 1048576 (1M tokens) | Full context window |
| **Chunked prefill** | Enabled | Splits long prompts across several steps |
| **Speculative decoding** | None (MTP is in the appendix) | Highest throughput at this batch size |
| **Reasoning parser** | `nemotron-v3` | Splits thinking from the final answer |
| **Tool parser** | `qwen3_coder` | Turns tool syntax into OpenAI `tool_calls` |
| **Host / port** | `127.0.0.1:8000` | Local-only, matches the client cells |

### Load the model

Run the following from inside the Docker terminal, in the same directory as `extra-llm-api-config.yml`.

> **Note:** The first launch takes a while. The progress readout can sit at `0%` for several minutes at a time: first while the weights download from Hugging Face and load, then while the server compiles CUDA kernels. That is expected rather than a hang, so give it time instead of restarting. Later launches reuse the cached weights and compiled kernels and start much faster.

> **Note:** Parser names are backend-specific: TensorRT-LLM's `nemotron-v3` is `nemotron_v3` in vLLM and `nemotron_3` in SGLang.

```shell
trtllm-serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --max_batch_size 8 \
  --max_num_tokens 8192 \
  --max_seq_len 1048576 \
  --trust_remote_code \
  --served_model_name nemotron-3.5-lightning \
  --reasoning_parser nemotron-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

### Wait for the server to be ready

In a new terminal, poll `/v1/models`:

```shell
until curl -sf http://localhost:8000/v1/models | grep -q nemotron-3.5-lightning; do
  echo "Waiting for server..."; sleep 10
done
echo "Server is ready"
```

Then check what the server is actually serving, and send one short request to confirm it generates:

```shell
curl -s http://localhost:8000/v1/models | python3 -m json.tool

curl -sf http://localhost:8000/v1/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "nemotron-3.5-lightning", "prompt": "Hello", "max_tokens": 16}'
```

> **Expected output:** The loop prints `Waiting for server...` while the model loads, then `Server is ready` once `nemotron-3.5-lightning` is listed by `/v1/models`. The model list shows that name as the `id`. The final command returns a short JSON completion, confirming the model is loaded and generating.

## Generate responses

The cells below show single, sequential, and streamed completions, followed by reasoning on/off, tool calling, and reasoning budget examples.

> **Note:** The reasoning trace and the final answer share one completion budget. If reasoning consumes all of `max_tokens`, the response comes back with `finish_reason: "length"` and `content` either empty or cut off mid-sentence. Raise `max_tokens` or shorten the prompt when that happens. The examples below budget a few thousand tokens for prompts that invite a long answer, and only a few hundred where reasoning is turned off.

> **Note:** `trtllm-serve` does not validate the `model` field on incoming requests, so a name that does not match the server is silently accepted rather than rejected.

### Client setup

In [3]:
from openai import OpenAI

# Set this to match the --served_model_name used when starting the server
SERVED_MODEL_NAME = "nemotron-3.5-lightning"
BASE_URL = "http://localhost:8000/v1"

client = OpenAI(base_url=BASE_URL, api_key="null")

### Single completion

In [4]:
# Single chat completion
response = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2048,
)
choice = response.choices[0]
print("Reasoning:", choice.message.reasoning_content)
print("Content:", choice.message.content)

Reasoning: Here's a thinking process:

1.  **Analyze User Request:**
   - User wants 3 bullet points about TensorRT-LLM
   - Format: 3 bullet points
   - Topic: TensorRT-LLM

2.  **Identify Key Concepts about TensorRT-LLM:**
   - What is TensorRT-LLM? It's an NVIDIA library for optimizing and deploying large language models (LLMs) on NVIDIA GPUs.
   - Key features: Performance optimization, ease of use, integration with PyTorch/Hugging Face, TensorRT engine generation, inference acceleration, support for various model architectures, quantization, kernel fusion, etc.
   - Main benefits: Speed, efficiency, reduced latency, seamless deployment, integration with popular frameworks.

3.  **Draft 3 Concise Bullet Points:**
   Need to capture the essence in 3 bullets. Let's make them informative but brief.

   Bullet 1: What it is/primary purpose. "TensorRT-LLM is NVIDIA's high-performance library for optimizing and deploying large language models on GPUs."
   Bullet 2: Key features/benefits.

### Sequential completions

Send multiple prompts in sequence and collect all responses.

In [5]:
prompts = [
    "What is the square root of 144?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
]

for prompt in prompts:
    response = client.chat.completions.create(
        model=SERVED_MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=1.0,
        top_p=0.95,
        max_tokens=2048,
    )
    print(f"Q: {prompt}")
    print(f"A: {response.choices[0].message.content}\n")

Q: What is the square root of 144?
A: The square root of 144 is **12** (since 12 × 12 = 144). If you're considering both roots, ±12 both square to 144, but the principal (non-negative) square root is 12.

Q: What is the capital of France?
A: The capital of France is Paris.

Q: Explain quantum computing in simple terms.
A: Here's a straightforward breakdown:

### Classical vs. Quantum
- **Classical computers** use **bits**: each is either `0` or `1`. They process information step-by-step, like reading a book from beginning to end.
- **Quantum computers** use **qubits** (quantum bits). Thanks to the physics of tiny particles, a qubit can be `0`, `1`, or **both at the same time**—until you measure it.

### The Two "Quantum Tricks"

**1. Superposition**  
Imagine a coin spinning on a table. While it's spinning, it's kind of both heads and tails at once. If you slap it down to look, it instantly becomes just one or the other.  
A qubit works similarly: while it's not being observed, it hold

### Streamed generation

Receive tokens as they are generated using the OpenAI streaming API.

In [6]:
# Streaming chat completion
print("Streaming response:")
stream = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    stream=True,
)

section = None

for chunk in stream:
    delta = chunk.choices[0].delta
    if not delta:
        continue

    reasoning = getattr(delta, "reasoning_content", None)
    if reasoning:
        if section != "reasoning":
            print("Reasoning: ", end="", flush=True)
            section = "reasoning"
        print(reasoning, end="", flush=True)

    if delta.content:
        if section != "content":
            print("\n\nContent: ", end="", flush=True)
            section = "content"
        print(delta.content, end="", flush=True)

Streaming response:
Reasoning: Here's a thinking process:

1.  **Analyze User Input:** User asks "What are the first 5 prime numbers?"
2.  **Identify Core Concept:** Prime numbers are natural numbers greater than 1 that have no positive divisors other than 1 and themselves.
3.  **Determine First 5 Primes:** 
   - 2 (smallest prime, only even prime)
   - 3
   - 5
   - 7
   - 11
   Let me verify: 
   - 2: prime ✓
   - 3: prime ✓
   - 4: not prime (divisible by 2)
   - 5: prime ✓
   - 6: not prime
   - 7: prime ✓
   - 8,9,10: not prime
   - 11: prime ✓
   So the first 5 are 2, 3, 5, 7, 11.
4.  **Formulate Response:** State them clearly. Maybe add a brief definition or note. Keep it concise.
   Output: The first 5 prime numbers are 2, 3, 5, 7, and 11.

Content: The first 5 prime numbers are **2, 3, 5, 7, and 11**.

(Note: A prime number is a natural number greater than 1 that has no positive divisors other than 1 and itself. 2 is also the only even prime number.)

### Reasoning

The model supports two modes: **Reasoning ON** (default) and **Reasoning OFF**.

Toggle by setting `enable_thinking` to `False` in `chat_template_kwargs`. Use `temperature=1.0, top_p=0.95` in both modes.

In [7]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=4096,
)
print("Reasoning:", resp.choices[0].message.reasoning_content)
print("Content:", resp.choices[0].message.content)
print()

# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 bullet points about TensorRT-LLM."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print("Content:", resp2.choices[0].message.content)

Reasoning on
Reasoning: Here, the user is asking for a haiku about GPUs. The user wants a haiku, a 17-syllable poem in three lines with a 5-7-5 syllable structure. I need to write a haiku about GPUs. I need to think of something related to GPUs: graphics cards, computing, parallel processing, gaming, deep learning, rendering, cores, etc. I need to make sure the syllable count is correct.

Let me brainstorm some possible lines:

Line 1 (5 syllables): Maybe "Silicon hearts" (2 syllables? Actually "Silicon" is 3, "hearts" is 1, total 4? Wait: Si-l-i-con = 3, hearts = 1, total 4. Need 5. Maybe "Chip of fire"? "Chip" is 1, "of" is 1, "fire" is 2? Actually "fire" is 1 syllable. So "Chip of fire" is 3 syllables. Not good.

Let's count syllables carefully.

I need 5 syllables in first line.

Possible 5-syllable words/phrases about GPUs:

- "Graphics card lights" -> Graphics (2), card (1), lights (1) = 4. Maybe "Graphic cards blaze" -> Graphic (2), cards (1), blaze (1) = 4.

- "Parallel cores h

### Tool calling

Call functions using the OpenAI Tools schema and inspect the returned `tool_calls`, then run the function and hand its result back so the model can answer the user.

> **Note:** When tool calling with reasoning enabled, pass `"force_nonempty_content": true` inside `chat_template_kwargs`. Without it, `content` can come back empty and the server may not surface the reasoning trace and the tool call together - coding agents in particular expect text alongside the call.

In [8]:
tools = [{
    "type": "function",
    "function": {
        "name": "calculate_tip",
        "description": "Calculate the tip amount for a bill",
        "parameters": {
            "type": "object",
            "properties": {
                "bill_total": {"type": "integer", "description": "The total amount of the bill"},
                "tip_percentage": {"type": "integer", "description": "The percentage of tip to apply"},
            },
            "required": ["bill_total", "tip_percentage"],
        },
    },
}]

messages = [{"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}]

completion = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages,
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    extra_body={"chat_template_kwargs": {"enable_thinking": True, "force_nonempty_content": True}},
)

choice = completion.choices[0]
print("Reasoning:", choice.message.reasoning_content)
print("Tool calls:", choice.message.tool_calls)

Reasoning: Here's a thinking process:

1.  **Analyze User Input:**
   - Bill total: $50
   - Tip percentage: 15%
   - Question: What will be the amount for 15% tip?

2.  **Identify Required Tool:**
   - The `calculate_tip` function takes `bill_total` and `tip_percentage` as integers.
   - `bill_total` = 50
   - `tip_percentage` = 15

3.  **Check Tool Parameters:**
   - `bill_total`: integer, required
   - `tip_percentage`: integer, required
   - Both match the user's values.

4.  **Call the Function:**
   - I will call `calculate_tip` with `bill_total=50` and `tip_percentage=15`.

5.  **Formulate Response:**
   - After getting the result, I'll state the tip amount and possibly the total amount if needed, but the question specifically asks "What will be the amount for 15% tip?" so I'll give the tip amount.

Let's call the function.✅

Tool calls: [ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-085e0e5ef25740f2b3b2e47704fbef82', function=Function(arguments='{"bill_total": 50, "ti

In [9]:
import json

def calculate_tip(bill_total, tip_percentage):
    """Return the tip and the final total for a bill."""
    tip = round(bill_total * tip_percentage / 100, 2)
    return {"tip": tip, "total": round(bill_total + tip, 2)}

# Map schema names to real functions, then run whichever one the model picked
tool_functions = {"calculate_tip": calculate_tip}

call = choice.message.tool_calls[0]
result = tool_functions[call.function.name](**json.loads(call.function.arguments))

# Hand the result back so the model can answer the user
followup = client.chat.completions.create(
    model=SERVED_MODEL_NAME,
    messages=messages + [
        choice.message,
        {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)},
    ],
    tools=tools,
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

print("Tool result:", result)
print("Final answer:", followup.choices[0].message.content)

Tool result: {'tip': 7.5, 'total': 57.5}
Final answer: The 15% tip on a $50 bill is **$7.50**, making the total amount **$57.50**.


### Controlling reasoning budget

The `reasoning_budget` parameter lets you limit how long the model reasons before producing a response. When the reasoning trace reaches the token budget, the model will try to wrap up at the next newline.

> **Note:** If no newline is encountered within 500 tokens after the budget threshold, the reasoning trace is forcibly terminated at `reasoning_budget + 500` tokens.

In [ ]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        response = self.client.chat.completions.create(
            model=model,
            messages=messages,
            max_tokens=reasoning_budget,
            **kwargs
        )

        reasoning_content = response.choices[0].message.reasoning_content or ""

        if "</think>" not in reasoning_content:
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"

        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used

        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )

        response = self.client.completions.create(
            model=model,
            prompt=prompt,
            max_tokens=remaining_tokens,
            **kwargs
        )

        return {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }

In [ ]:
budget_client = ThinkingBudgetClient(
    base_url="http://localhost:8000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4"  # use actual HF model ID for tokenizer
)

In [12]:
resp = budget_client.chat_completion(
    model=SERVED_MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=1024,
    reasoning_budget=128
)
print("Reasoning:", resp["reasoning_content"])
print("Content:", resp["content"])

Reasoning: Here's a thinking process:

1.  **Analyze the Request**: The user wants a haiku about GPUs.
2.  **Understand the Constraints**: 
   - A haiku is a traditional Japanese poem with a 5-7-5 syllable structure (total 17 syllables).
   - The topic is GPUs (Graphics Processing Units).
3.  **Brainstorm GPU-related concepts/words**: 
   - Graphics, processors, cores, chips, speed, render, pixels, screens, compute, cores, parallel, frames, gaming, power, heat, silicon, etc.
4.
Content: Silicon cores hum bright,
Rendering worlds in electric glow,
Frames rise like sunrise.


## Cleanup and shutdown

To tear down this TensorRT-LLM workflow:

1. In the terminal running `trtllm-serve`, press `Ctrl+C` to stop the server.
2. In the Docker shell, run `exit` to stop the container (`--rm` removes it automatically).

## Additional configurations

Reference configurations for the combinations this notebook does not run.

Each entry gives a full `extra-llm-api-config.yml`, followed by the block that switches on a speculator. Add that block to the YAML file - it is a fragment and is not a complete config on its own. TensorRT-LLM supports MTP only - DFlash and DSpark are covered in the vLLM and SGLang cookbooks.

### 1x H100

#### NVFP4

**Base**: this is the primary path above, under **Create a YAML file**.

**Add MTP**

This checkpoint has a single MTP layer, applied repeatedly to produce the 3 draft tokens set by `max_draft_len` - the layer count comes from the checkpoint and is not configurable. Add the block alongside the other top-level keys in the YAML and leave the rest of it unchanged.

```yaml
speculative_config:
  decoding_type: MTP
  max_draft_len: 3
  allow_advanced_sampling: true
```

#### BF16

**Base**

BF16 runs through the CUTLASS MoE backend and serves a 256K context, which is what fits one 80GB H100 at this precision.

```shell
cat > ./extra-llm-api-config.yml << EOF
kv_cache_config:
  dtype: fp8
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.8
  mamba_ssm_cache_dtype: float16
  mamba_ssm_stochastic_rounding: true
  mamba_ssm_philox_rounds: 5
  mamba_state_config:
    periodic_snapshot_interval: 8192

moe_config:
  backend: CUTLASS

cuda_graph_config:
  enable_padding: true
  max_batch_size: 32

enable_chunked_prefill: true
num_postprocess_workers: 4
print_iter_log: true
stream_interval: 10
disable_overlap_scheduler: false
EOF
```

```shell
trtllm-serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
  --host 127.0.0.1 \
  --port 8000 \
  --max_batch_size 32 \
  --max_num_tokens 8192 \
  --max_seq_len 262144 \
  --trust_remote_code \
  --served_model_name nemotron-3.5-lightning \
  --reasoning_parser nemotron-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

**Add MTP**

This checkpoint has a single MTP layer, applied repeatedly to produce the 3 draft tokens set by `max_draft_len` - the layer count comes from the checkpoint and is not configurable. Add the block alongside the other top-level keys in the YAML and leave the rest of it unchanged.

```yaml
speculative_config:
  decoding_type: MTP
  max_draft_len: 3
  allow_advanced_sampling: true
```

### 1x DGX Spark

#### NVFP4

**Base**

DGX Spark uses the CUTEDSL MoE backend and serves a single request at a time at a 1M-token context.

```shell
cat > ./extra-llm-api-config.yml << EOF
kv_cache_config:
  dtype: fp8
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.8
  mamba_ssm_cache_dtype: float16
  mamba_ssm_stochastic_rounding: true
  mamba_ssm_philox_rounds: 5
  mamba_state_config:
    periodic_snapshot_interval: 8192

moe_config:
  backend: CUTEDSL

nvfp4_gemm_config:
  allowed_backends: [marlin, cutlass, cublaslt, cuda_core]

cuda_graph_config:
  enable_padding: true
  max_batch_size: 16

enable_chunked_prefill: true
num_postprocess_workers: 4
print_iter_log: true
stream_interval: 10
disable_overlap_scheduler: false
EOF
```

```shell
trtllm-serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-NVFP4 \
  --host 127.0.0.1 \
  --port 8000 \
  --max_batch_size 1 \
  --max_num_tokens 8192 \
  --max_seq_len 1048576 \
  --trust_remote_code \
  --served_model_name nemotron-3.5-lightning \
  --reasoning_parser nemotron-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

**Add MTP**

This checkpoint has a single MTP layer, applied repeatedly to produce the 3 draft tokens set by `max_draft_len` - the layer count comes from the checkpoint and is not configurable. Add the block alongside the other top-level keys in the YAML and leave the rest of it unchanged.

```yaml
speculative_config:
  decoding_type: MTP
  max_draft_len: 3
  allow_advanced_sampling: true
```

#### BF16

**Base**

BF16 runs through the CUTLASS MoE backend here.

```shell
cat > ./extra-llm-api-config.yml << EOF
kv_cache_config:
  dtype: fp8
  enable_block_reuse: false
  free_gpu_memory_fraction: 0.8
  mamba_ssm_cache_dtype: float16
  mamba_ssm_stochastic_rounding: true
  mamba_ssm_philox_rounds: 5
  mamba_state_config:
    periodic_snapshot_interval: 8192

moe_config:
  backend: CUTLASS

cuda_graph_config:
  enable_padding: true
  max_batch_size: 16

enable_chunked_prefill: true
num_postprocess_workers: 4
print_iter_log: true
stream_interval: 10
disable_overlap_scheduler: false
EOF
```

```shell
trtllm-serve nvidia/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-BF16 \
  --host 127.0.0.1 \
  --port 8000 \
  --max_batch_size 1 \
  --max_num_tokens 8192 \
  --max_seq_len 1048576 \
  --trust_remote_code \
  --served_model_name nemotron-3.5-lightning \
  --reasoning_parser nemotron-v3 \
  --tool_parser qwen3_coder \
  --extra_llm_api_options extra-llm-api-config.yml
```

**Add MTP**

This checkpoint has a single MTP layer, applied repeatedly to produce the 3 draft tokens set by `max_draft_len` - the layer count comes from the checkpoint and is not configurable. Add the block alongside the other top-level keys in the YAML and leave the rest of it unchanged.

```yaml
speculative_config:
  decoding_type: MTP
  max_draft_len: 3
  allow_advanced_sampling: true
```